In [1]:
import pandas as pd
import numpy as np
from langchain_chroma import Chroma

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI


from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


import os
from dotenv import load_dotenv
load_dotenv()



USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [2]:
# Loading the keys

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

### Performing similarity search using the Ollama Embeddings and Chroma DB

In [ ]:
#### Storing and retrieving from the databases, and performing a similarity search
ollama_embeddings=(
    OllamaEmbeddings(model="gemma:2b")  ##by default it ues llama2. gemma:2b is downloaded in local pc
)

# Loading the research paper

pdf_loader = PyPDFLoader('ip_data/attention-is-all-you-need.pdf')
docs = pdf_loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_documents=text_splitter.split_documents(docs)

# Storing the documents in the chroma db
# Always need to pass the embeddings and documents together to save them.
vector_db = Chroma.from_documents(final_documents,ollama_embeddings, persist_directory = 'stored_data\chroma_db')

<>:14: SyntaxWarning: invalid escape sequence '\c'
<>:14: SyntaxWarning: invalid escape sequence '\c'
C:\Users\sagnik\AppData\Local\Temp\ipykernel_27016\63906354.py:14: SyntaxWarning: invalid escape sequence '\c'
  vector_db = Chroma.from_documents(final_documents,ollama_embeddings, persist_directory = 'stored_data\chroma_db')


In [40]:
# Performing a simimarity search from the chroma db store : 
query = "How exactly the self-attention mechanism works?"
docs_from_query = vector_db.similarity_search(query)
print(docs_from_query)

for i, doc in enumerate(docs_from_query):
    print(f"Result {i+1}")
    print(f"Content:\n{doc.page_content.strip()}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)
# How to get the docs in a better format?
# Use the query in one of the chatbots to load the stuff chain document


[Document(id='b91e0727-de82-4b46-a592-ba21038e75f1', metadata={'page': 1, 'source': 'ip_data/attention-is-all-you-need.pdf'}, page_content='described in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, 22, 23, 19].'), Document(id='a2b47ed4-c9ab-4888-b185-cb42576fd077', metadata={'source': 'ip_data/attention-is-all-you-need.pdf', 'page': 1}, page_content='described in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comp

### Retriever and Chain for Ollama

In [4]:
loader=WebBaseLoader("https://en.wikipedia.org/wiki/Boeing#Criticism")
docs = loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [ ]:
llm = llm = Ollama(model="gemma:2b")

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})


[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'ğŸ¦œï¸�ğŸ›\xa0ï¸� LangSmith', 'language': 'en'}, page_content='\n\n\n\n\nğŸ¦œï¸�ğŸ›\xa0ï¸� LangSmith\n\n\n\n\n\n\n\n\nSkip to main contentOur Building Ambient Agents with LangGraph course is now available on LangChain Academy!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppPage Not FoundWe could not find what you were looking for.Head back to our main docs page or use the search bar to find the page you need.CommunityLangChain ForumTwitterGitHubDocs CodeLangSmith SDKPythonJS/TSMoreHomepageBlogLangChain Python DocsLangChain JS/TS DocsCopyright Â© 2025 LangChain, Inc.\n\n')]

In [5]:
for i, doc in enumerate(docs):
    print(f"Result {i+1}")
    print(f"Content:\n{doc.page_content.strip()}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)
    if i == 10:
        break

Result 1
Content:
Boeing - Wikipedia


































Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





Donate Create account Log in





		Pages for logged out editors learn more



ContributionsTalk




























Contents
move to sidebar
hide




(Top)





1
History




Toggle History subsection





1.1
Origins








1.2
Sea Launch








1.3
Merger with McDonnell Douglas








1.4
Corporate headquarters moves








1.5
Labor strike










2
Divisions








3
Safety defects and airplane crashes




Toggle Safety defects and airplane crashes subsection





3.1
Boeing 737 MAX crashes and gro

### Building a chatbot through OpenAI

### Building a chatbot through Hugging Face

### Building a chatbot through GroqAPI

#### Storing things in a single database - select your own